# Pipeline Mini-4K → dataset de México → fine-tuning SAM-TP

Notebook orquestador. **No reimplementa nada**: llama a los scripts que ya están
en `ML_model/scripts/pipe_videos_online/`.

Guardalo en `ML_model/notebooks/` (o donde quieras: la celda 0 encuentra la raíz sola).

Orden: 0 setup → 1 metadata → 2 explorar → 3 select → 4 fetch → 5 clean → 6 curate → 7 chequeo.
Corré una celda a la vez y mirá la salida antes de seguir.


## 0 — Setup de rutas

El notebook vive en `ML_model/notebooks/`, asi que la raiz es `..`.
Todas las rutas salen de ahi. No cambia el directorio de trabajo.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

In [ ]:


# Este notebook vive en ML_model/notebooks/  ->  la raiz es la carpeta de arriba.

ML_ROOT   = Path.cwd().parent
PIPE_DIR  = ML_ROOT / "scripts" / "pipe_videos_online"
RIDES_DIR = ML_ROOT / "scripts" / "pipe_nuestras_rides"
DATA      = ML_ROOT / "data"
META_DIR  = ML_ROOT / "frodobots_metadata"      # gitignored
WORK      = DATA / "mexico"                     # todo lo de esta corrida vive aca
WORK.mkdir(parents=True, exist_ok=True)

# archivos que produce el pipeline, todos bajo WORK
SELECTED  = WORK / "selected_mexico.csv"
RAW_DIR   = WORK / "raw"
CLEAN_DIR = WORK / "clean"
CANDIDATOS = WORK / "candidatos"
INDEX     = WORK / "cleaned_index.csv"
CURATED   = WORK / "curated_mexico.csv"

sys.path.insert(0, str(PIPE_DIR))   # para importar Tool_2_Revisar_Parquet

assert PIPE_DIR.is_dir(), f"No existe {PIPE_DIR} -- corre el notebook desde ML_model/notebooks/"

for k, v in dict(ML_ROOT=ML_ROOT, PIPE_DIR=PIPE_DIR, META_DIR=META_DIR, WORK=WORK).items():
    print(f"{k:9s} = {v}   {'OK' if Path(v).exists() else '(no existe todavia)'}")


In [ ]:
def run(script: str, *args, cwd: Path = None, python: str = None) -> int:
    """Corre un script del repo mostrando la salida en vivo.

    `python`: interprete a usar (default: sys.executable, el del kernel).
    Pasalo cuando el script necesite un entorno DISTINTO al del notebook --
    por ejemplo 2_evaluar_con_modelo.py, que corre SAM-TP de verdad (torch,
    hydra, sam2) y por eso necesita el mismo entorno que usa bridge.py, no
    el .venv-data liviano de ML_model.
    """
    cwd = cwd or PIPE_DIR
    exe = python or sys.executable
    cmd = [exe, script, *[str(a) for a in args]]
    print("$", " ".join(cmd), f"   (cwd={cwd})\n")
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[exit {p.returncode}]")
    return p.returncode


## 1 — Metadata del Mini-4K

Primero **listamos** los archivos del repo (celda 1a), despues **bajamos**
solo los livianos (celda 1b). Nada de video.

> Los patrones de `allow_patterns` van **sin** `**/`: en fnmatch el `*` ya
> cruza las barras, y `**/*.parquet` exige una barra literal, por lo que se
> saltea `metadata.parquet` si esta en la raiz del repo.


In [ ]:
# 1a — Ver que hay realmente en el repo (no baja nada, solo lista nombres)
from huggingface_hub import list_repo_files, snapshot_download

files = list_repo_files("BitRobot/FrodoBots-Mini-4K", repo_type="dataset")
print(f"{len(files)} archivos en el repo\n")

livianos = [f for f in files if f.endswith((".parquet", ".json", ".csv", ".md", ".txt"))]
print(f"{len(livianos)} archivos livianos (parquet/json/csv/md):")
for f in livianos[:60]:
    print("  ", f)
if len(livianos) > 60:
    print(f"   ... y {len(livianos)-60} mas")

# los parquets, que es lo que nos importa
print("\nParquets:")
for f in [f for f in files if f.endswith(".parquet")][:30]:
    print("  ", f)


In [ ]:
# 1b — Bajar los archivos livianos
#
# OJO con los patrones: huggingface_hub filtra con fnmatch, donde "*" YA cruza
# las barras. Por eso "**/*.parquet" exige una barra literal y NO matchea un
# archivo en la raiz como "metadata.parquet". Los patrones van sin "**/".
#
# Mira la salida de la celda 1a y ajusta ALLOW si hace falta:
#   - si hay UN solo parquet de metadata en la raiz -> ["metadata.parquet"]
#   - si hay varios y los queres todos             -> ["*.parquet", "*.json", "*.csv"]

ALLOW = ["*.parquet", "*.json", "*.csv"]     # <-- EDITAR segun la celda 1a

if not any(META_DIR.rglob("*.parquet")):
    snapshot_download(
        repo_id="BitRobot/FrodoBots-Mini-4K",
        repo_type="dataset",
        allow_patterns=ALLOW,
        local_dir=str(META_DIR),
    )
else:
    print("Ya hay parquets en", META_DIR, "-- borra la carpeta si queres rebajarlos")

parquets = sorted(META_DIR.rglob("*.parquet"))
print(f"\n{len(parquets)} parquets locales:")
for p in parquets[:20]:
    print("  ", p.relative_to(META_DIR), f"{p.stat().st_size/1e6:.1f} MB")

if not parquets:
    raise SystemExit(
        "No se bajo ningun parquet. Revisa ALLOW contra los nombres reales "
        "que imprimio la celda 1a. Si el dataset es gated, corre "
        "'huggingface-cli login' primero."
    )

META_PARQUET = next((p for p in parquets if p.name == "metadata.parquet"), parquets[0])
print("\nMETA_PARQUET =", META_PARQUET)


## 2 — Explorar el metadata y encontrar el nombre exacto de México

Usa tu propio `Tool_2_Revisar_Parquet.explorar_parquet`. El filtro del `select`
es por coincidencia exacta, así que acá confirmás si dice `Mexico`, `México` o `MX`.

In [ ]:
from Tool_2_Revisar_Parquet import explorar_parquet

df_meta = explorar_parquet(META_PARQUET)

print("\n--- paises ---")
print(df_meta["country"].value_counts().to_string())


In [ ]:
# Elegí el valor EXACTO tal como aparece arriba
PAIS = "Mexico"     # <-- EDITAR si el value_counts dice otra cosa

mx = df_meta[df_meta["country"] == PAIS]
print(f"{len(mx)} rides en {PAIS}, {mx['db_dur_sec'].sum()/3600:.1f} h totales\n")

flags = [c for c in ["has_control","has_gps","has_imu","has_front_ts",
                     "has_rear_camera","has_video","complete"] if c in mx.columns]
print("Cuantos rides cumplen cada condicion:")
print(mx[flags].sum().to_string())

# cuantos pasan TODOS los filtros que exige el select
base = mx[flags[:4]].all(axis=1) if len(flags) >= 4 else None
if base is not None:
    print(f"\nPasan control+gps+imu+front_ts: {base.sum()}")
    if "has_rear_camera" in mx.columns:
        print(f"  ...y ademas tienen camara trasera: {(base & mx['has_rear_camera']).sum()}")


In [ ]:
# Inventario completo a Excel, por si lo querés mirar a mano
xlsx = WORK / "metadata_mexico.xlsx"
with __import__("pandas").ExcelWriter(xlsx, engine="openpyxl") as xl:
    mx.head(1_000_000).to_excel(xl, sheet_name="rides_mexico", index=False)
    df_meta["country"].value_counts().rename("n_rides").to_frame().to_excel(xl, sheet_name="por_pais")
print("->", xlsx)


## 3 — `select`: filtrar rides de México (no baja video)

Si el resultado te da pocos rides, sacá `--require-rear` y volvé a correr.

In [ ]:
REQUIRE_REAR = True    # <-- poné False si el paso 2 mostro pocos rides con camara trasera

args = ["--countries", PAIS, "--out", SELECTED]
if REQUIRE_REAR:
    args.append("--require-rear")

run("1_descarga_filtrado.py", "select", *args)


In [ ]:
import pandas as pd

sel = pd.read_csv(SELECTED)
print(f"{len(sel)} rides, {sel['db_dur_sec'].sum()/3600:.1f} h")
sel.to_excel(SELECTED.with_suffix(".xlsx"), index=False)
print("Excel ->", SELECTED.with_suffix(".xlsx"))
sel.head(20)


## 4 — `fetch`: bajar solo esos rides

**Este paso sí baja datos pesados.** Baja los shards que contienen los rides elegidos
y extrae solo esos. Empezá con pocos rides para medir cuánto tarda y cuánto ocupa.

In [ ]:
# Opcional: recortar a los N rides mas largos para una primera prueba
N_PRUEBA = 10          # <-- None para bajar todos los seleccionados

rides_file = SELECTED
if N_PRUEBA:
    rides_file = WORK / f"selected_mexico_top{N_PRUEBA}.csv"
    sel.head(N_PRUEBA).to_csv(rides_file, index=False)
    print(f"Usando los {N_PRUEBA} rides mas largos -> {rides_file}")

run("1_descarga_filtrado.py", "fetch", "--rides", rides_file, "--raw-dir", RAW_DIR)


In [ ]:
# Cuanto ocupo lo que bajaste
total = sum(p.stat().st_size for p in RAW_DIR.rglob("*") if p.is_file())
print(f"{RAW_DIR}: {total/1e9:.2f} GB en {len(list(RAW_DIR.iterdir()))} carpetas de ride")


## 5 — `clean`: sincronizar frame + GPS + control + IMU

Tolerancia de 500 ms, descarta GPS sin fix. Deja un `.synced.parquet` por ride
y un índice CSV.

In [ ]:
run("1_descarga_filtrado.py", "clean",
    "--raw-dir", RAW_DIR, "--clean-dir", CLEAN_DIR, "--index", INDEX)


In [ ]:
import pandas as pd
idx = pd.read_csv(INDEX)
print(idx.describe(include="all").T.to_string())
idx.head(20)


## 6 — `curate`: deduplicar por celda GPS

Evita quedarte con 30 rides de la misma cuadra. Para un primer fine-tune,
unas pocas horas bien diversas rinden más que muchas redundantes.

In [ ]:
ride_folder = "ride_111444_cpgq1m_20250331131226"
for p in sorted((RAW_DIR / ride_folder).rglob("*")):
    print(p.relative_to(RAW_DIR / ride_folder))

In [ ]:
HORAS = 20      # <-- objetivo de horas; bajalo para el primer fine-tune

run("1_descarga_filtrado.py", "curate",
    "--index", INDEX, "--hours", HORAS, "--out", CURATED)


In [ ]:
cur = pd.read_csv(CURATED)
cur.to_excel(CURATED.with_suffix(".xlsx"), index=False)
print(f"{len(cur)} rides curados -> {CURATED.with_suffix('.xlsx')}")
cur.head(20)


## 7 — Extraer frames de los rides curados

`curate` eligió *qué rides* usar, pero todavía son videos, no imágenes. Este
paso usa `1b_frames_desde_rides.py` (no viene en el repo original — hay que
copiarlo a `scripts/pipe_videos_online/` una sola vez) para sacar un frame
cada `EVERY_N_M` metros de distancia GPS real, por cada ride curado.

Corré primero la celda de `--inspeccionar` para confirmar los nombres de
columnas de tu `.synced.parquet` antes de procesar todo — si el dataset trae
otros nombres de columna, pasalos con `--col-lat/--col-lon/--col-ts`.

In [ ]:
import cv2
CANDIDATOS = WORK / "candidatos"
CANDIDATOS.mkdir(parents=True, exist_ok=True)


In [ ]:
EVERY_N_M     = 5     # <-- espaciado entre frames elegidos, en metros
MAX_POR_RIDE  = 200     # <-- tope de frames por ride, para no desbalancear

run("1b_frames_desde_rides.py",
    "--clean-dir", CLEAN_DIR, "--raw-dir", RAW_DIR, "--rides", CURATED,
    "--out", CANDIDATOS,  "--max-por-ride", MAX_POR_RIDE)

In [ ]:
n_frames = len(list(CANDIDATOS.glob("*_rgb.jpg")))
print(f"{n_frames} frames extraidos -> {CANDIDATOS}")

## 8 — Pre-anotar con el modelo actual

Corre el checkpoint de hoy sobre los candidatos: genera una máscara
propuesta por frame (`--guardar-mascaras`) y un puntaje de incertidumbre en
`model_eval.jsonl`. Los frames de mayor incertidumbre son los que conviene
etiquetar primero — normalmente ahí está concentrada la confusión
pasto/camino que estamos tratando de arreglar.

In [ ]:
from pathlib import Path

carpeta = Path("/home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos_run")

for archivo in carpeta.rglob("*"):
    if archivo.is_file() and archivo.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        
        # Si ya termina en rgb.jpg, no hacer nada
        if archivo.name.endswith("rgb.jpg"):
            continue

        # Crear nuevo nombre terminando en rgb.jpg
        nuevo_nombre = archivo.stem + "_rgb.jpg"
        nuevo_path = archivo.with_name(nuevo_nombre)

        # Evitar sobrescribir archivos existentes
        contador = 1
        while nuevo_path.exists():
            nuevo_nombre = f"{archivo.stem}_{contador}_rgb.jpg"
            nuevo_path = archivo.with_name(nuevo_nombre)
            contador += 1

        print(f"Renombrando:\n{archivo}\n→ {nuevo_path}\n")
        archivo.rename(nuevo_path)

print("Proceso terminado.")

In [ ]:
NAV_REPO = ML_ROOT.parent
assert (NAV_REPO / "genie").is_dir(), (
    f"No encontre {NAV_REPO / 'genie'} -- eso significa que ML_ROOT ({ML_ROOT}) "
    "no es realmente IROS26-LaRovernetta/ML_model. Revisa desde donde corriste "
    "el notebook (tiene que vivir en <repo>/ML_model/notebooks/)."
)
os.environ["NAV_REPO_PATH"] = str(NAV_REPO)   # para que config_paths.py encuentre genie/

BRIDGE_CONFIG = NAV_REPO / "genie" / "configs" / "frodobot_rover.yaml"
assert BRIDGE_CONFIG.is_file(), (
    f"No encontre el config del bridge en {BRIDGE_CONFIG} -- confirma que ese "
    "es el yaml vigente (y no un .yaml.save/backup)."
)

# Ruta real del checkpoint (checkpoint_2.pt, el que declara samtp.checkpoint_path
# en frodobot_rover.yaml -- se pasa explicito por --checkpoint para no depender
# de que el yaml lo resuelva bien).
CHECKPOINT_REAL = (
    NAV_REPO / "genie" / "sam2_logs" / "configs"
    / "sam2.1_training_tiny" / "sam2_training_custom2_freezeNoneNone_f57.yaml"
    / "checkpoints" / "checkpoint_2.pt"
)
assert CHECKPOINT_REAL.is_file(), f"No encontre el checkpoint en {CHECKPOINT_REAL}"

# Ruta real del config de SAM-TP (samtp.config_path del mismo yaml).
SAMTP_CONFIG_REAL = (
    NAV_REPO / "genie" / "sam2" / "configs"
    / "sam2.1_inference_tiny" / "sam2.1_custom2.yaml"
)
assert SAMTP_CONFIG_REAL.is_file(), f"No encontre el config de SAM-TP en {SAMTP_CONFIG_REAL}"

# 2_evaluar_con_modelo.py ya no usa rover_traversability, pero SI carga SAM-TP
# de verdad (torch, hydra, sam2) -- el .venv-data de ML_model es liviano a
# proposito y no tiene esas libs instaladas. Necesita el MISMO entorno que
# usa bridge.py (el que tiene torch+hydra+sam2).
# <-- EDITAR con la ruta real: activa ese entorno y corre `which python`.
GENIE_PYTHON = "/home/pablolube/IROS26-LaRovernetta/genie/.venv/bin/python"
assert Path(GENIE_PYTHON).is_file(), (
    f"No encontre el interprete en {GENIE_PYTHON} -- editá GENIE_PYTHON con la ruta "
    "real de tu entorno de genie (el mismo que usás para correr bridge.py)."
)

EVAL_OUT      = CANDIDATOS / "model_eval.jsonl2"
OVERLAYS_DIR  = CANDIDATOS / "overlays2"

run("2_evaluar_con_modelo.py",
    "--frames-dir","/home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos_run/",
    "--out", EVAL_OUT,
    "--overlays-dir", OVERLAYS_DIR,
    "--guardar-mascaras",
    "--bridge-config", BRIDGE_CONFIG,
    "--checkpoint", CHECKPOINT_REAL,
    "--samtp-config", SAMTP_CONFIG_REAL,
    cwd=RIDES_DIR, python=GENIE_PYTHON)

In [ ]:
EVAL_BEV_OUT = CANDIDATOS / "perception_bev_eval.jsonl"
DEBUG_BEV_DIR = CANDIDATOS / "debug_bev"

run("2c_evaluar_perception_bev.py",
    "--frames-dir", "/home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos_run/",
    "--out", EVAL_BEV_OUT,
    "--debug-dir", DEBUG_BEV_DIR,
    "--bridge-config", BRIDGE_CONFIG,
    "--checkpoint", CHECKPOINT_REAL,
    "--samtp-config", SAMTP_CONFIG_REAL,
    cwd=RIDES_DIR, python=GENIE_PYTHON)

$ /home/pablolube/IROS26-LaRovernetta/genie/.venv/bin/python 2c_evaluar_perception_bev.py --frames-dir /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos_run/ --out /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos/perception_bev_eval.jsonl --debug-dir /home/pablolube/IROS26-LaRovernetta/ML_model/data/mexico/candidatos/debug_bev --bridge-config /home/pablolube/IROS26-LaRovernetta/genie/configs/frodobot_rover.yaml --checkpoint /home/pablolube/IROS26-LaRovernetta/genie/sam2_logs/configs/sam2.1_training_tiny/sam2_training_custom2_freezeNoneNone_f57.yaml/checkpoints/checkpoint_2.pt --samtp-config /home/pablolube/IROS26-LaRovernetta/genie/sam2/configs/sam2.1_inference_tiny/sam2.1_custom2.yaml    (cwd=/home/pablolube/IROS26-LaRovernetta/ML_model/scripts/pipe_nuestras_rides)

/home/pablolube/IROS26-LaRovernetta/genie/.venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple ver

In [ ]:
import pandas as pd
# Prioridad de etiquetado: ordenado por incertidumbre descendente.
eval_df = pd.read_json(EVAL_OUT, lines=True).sort_values("incertidumbre", ascending=False)
eval_df.to_excel(EVAL_OUT.with_suffix(".xlsx"), index=False)
print(f"{len(eval_df)} frames evaluados -> {EVAL_OUT.with_suffix('.xlsx')}")
eval_df.head(20)

In [ ]:
# Cuantos frames corregir a mano -- ajustable segun tiempo disponible del equipo
N_A_ETIQUETAR = 5

a_etiquetar = eval_df.head(N_A_ETIQUETAR)
print(f"Top {len(a_etiquetar)} frames por incertidumbre -- son los que hay que "
      f"corregir a mano en el paso 9.\nMirar primero los overlays en {OVERLAYS_DIR} "
      f"para confirmar que la incertidumbre alta tiene sentido a ojo.")
a_etiquetar[["frame_file", "incertidumbre", "drivable_frac"]].head(10)

## 9 — Corregir las máscaras (trabajo manual del equipo)

Esto **no se automatiza** — es el paso donde una persona revisa cada
`<id>_mask.png` pre-generado (dentro de `data/mexico/candidatos/`) contra su
`<id>_rgb.jpg`, con `2b_corregir_mascaras.py`, CVAT o Label Studio.

**Regla única para todo el equipo:** pasto es *siempre* no transitable
(blanco = transitable, negro = no), aunque esté pisado y parezca senda. Si
cada persona traza el borde distinto, el modelo aprende la ambigüedad que
justamente queremos sacarle.

Priorizá los `N_A_ETIQUETAR` frames de la celda anterior — son donde el
modelo actual duda más.

## 10 — (Opcional) Mezclar con frames del dataset viejo

Si entrenás **solo** con frames de México, el modelo mejora ahí y empieza a
fallar en escenas que hoy resuelve bien (*catastrophic forgetting*). Si
tenés acceso a una muestra del dataset con el que se entrenó
`checkpoint_finetuned_v2.pt` (con sus máscaras ya buenas), copiá ~25% de esa
cantidad de pares acá adentro de `CANDIDATOS` **antes** de armar el dataset
final en el paso 11.

Dejalo en `None` si todavía no tenés esos frames a mano — podés agregarlos
más tarde y volver a correr el paso 11.

In [ ]:
import shutil

OLD_FRAMES_DIR = None   # <-- poné la ruta a pares *_rgb.jpg/*_mask.png viejos, o dejalo en None

if OLD_FRAMES_DIR:
    OLD_FRAMES_DIR = Path(OLD_FRAMES_DIR)
    pares_viejos = sorted(OLD_FRAMES_DIR.glob("*_rgb.jpg"))
    objetivo = max(1, int(0.25 * len(list(CANDIDATOS.glob('*_mask.png')))))
    copiados = 0
    for rgb in pares_viejos:
        mask = rgb.with_name(rgb.name.replace("_rgb.jpg", "_mask.png"))
        if not mask.exists():
            continue
        shutil.copy2(rgb, CANDIDATOS / rgb.name)
        shutil.copy2(mask, CANDIDATOS / mask.name)
        copiados += 1
        if copiados >= objetivo:
            break
    print(f"{copiados} pares viejos copiados a {CANDIDATOS} (objetivo ~25% -> {objetivo})")
else:
    print("OLD_FRAMES_DIR no seteado -- salteando la mezcla. "
          "Recorda que sin mezcla el riesgo de catastrophic forgetting es alto.")

## 11 — Armar el dataset final

Toma los pares ya corregidos a mano (`<id>_rgb.jpg` + `<id>_mask.png`) y
arma el layout estilo MOSE que espera el training config de SAM2:
`img_folder/{train,val}/<id>/00000.jpg` + `gt_folder/{train,val}/<id>/00000.png`.

In [ ]:
FOLD_DIR = ML_ROOT / "training" / "FOLD_mexico"

run("3_armar_dataset.py",
    "--labeled", CANDIDATOS, "--out", FOLD_DIR, "--val-frac", "0.15",
    cwd=RIDES_DIR)

# ENTRENAMIENTO!

## 12 — Preparar el config de entrenamiento

El loop de entrenamiento no vive en este repo — se clona
`facebookresearch/sam2` aparte. Acá copiamos nuestro config
(`genie/sam2/configs/sam2.1_training_tiny/sam2.1_custom2.yaml`) a una
versión nueva (`sam2.1_custom2_mexico.yaml`) y **mostramos** qué líneas hay
que apuntar a `FOLD_mexico` y al checkpoint actual.

⚠️ La estructura exacta del YAML (nombres de claves anidadas) puede variar
según la versión del config — la celda de abajo hace el reemplazo por texto
en las líneas que contienen `img_folder:`, `gt_folder:`, `checkpoint_path:`
y `max_epochs:`, e imprime el diff **antes** de guardar. Revisalo a ojo
contra el archivo real antes de confirmar.

In [ ]:
SAM2_CLONE_DIR = Path.home() / "sam2"   # <-- ajustar a donde clonaste facebookresearch/sam2
CONFIG_SRC = ML_ROOT / "genie" / "sam2" / "configs" / "sam2.1_training_tiny" / "sam2.1_custom2.yaml"
CONFIG_DST = SAM2_CLONE_DIR / "sam2" / "configs" / "sam2.1_training_tiny" / "sam2.1_custom2_mexico.yaml"

assert CONFIG_SRC.exists(), f"No encontre el config base en {CONFIG_SRC}"
assert SAM2_CLONE_DIR.exists(), (
    f"No encontre el clone de sam2 en {SAM2_CLONE_DIR} -- "
    f"'git clone https://github.com/facebookresearch/sam2.git' primero"
)

CONFIG_DST.parent.mkdir(parents=True, exist_ok=True)
texto_original = CONFIG_SRC.read_text()
print(texto_original)

In [ ]:
import re

CHECKPOINT_PATH = "checkpoint_finetuned_v2.pt"   # <-- ruta local a tu checkpoint actual
MAX_EPOCHS = 3                                    # <-- 2-3 para un fine-tune chico, nunca 5

reemplazos = {
    r"(img_folder:\s*).*":      lambda m: m.group(1) + str(FOLD_DIR / "train" / "img_folder"),
    r"(gt_folder:\s*).*":       lambda m: m.group(1) + str(FOLD_DIR / "train" / "gt_folder"),
    r"(checkpoint_path:\s*).*": lambda m: m.group(1) + CHECKPOINT_PATH,
    r"(max_epochs:\s*).*":      lambda m: m.group(1) + str(MAX_EPOCHS),
}

texto_nuevo = texto_original
cambios = []
for patron, repl in reemplazos.items():
    nuevo = re.sub(patron, repl, texto_nuevo)
    if nuevo != texto_nuevo:
        cambios.append(patron)
    texto_nuevo = nuevo

print("Claves encontradas y reemplazadas:", cambios or "NINGUNA -- revisa los nombres de clave a mano")
print("\n--- resultado propuesto ---\n")
print(texto_nuevo)

In [ ]:
GUARDAR_CONFIG = False   # <-- poné True recien despues de revisar el diff de arriba

if GUARDAR_CONFIG:
    CONFIG_DST.write_text(texto_nuevo)
    print(f"Config guardado -> {CONFIG_DST}")
else:
    print("GUARDAR_CONFIG=False -- no se escribio nada todavia")

## 13 — Lanzar el entrenamiento

Requiere GPU real (resolución 1024 no entra en CPU ni en laptops chicas). En
una GPU de 8 GB bajá `batch_size` en el config a 1-2 antes de correr esto.

`EJECUTAR_ENTRENAMIENTO` arranca en `False` a propósito — un entrenamiento
real tarda horas y no es algo para disparar por accidente re-corriendo el
notebook entero.

In [ ]:
EJECUTAR_ENTRENAMIENTO = False   # <-- poné True cuando quieras lanzarlo de verdad

if EJECUTAR_ENTRENAMIENTO:
    run("training/train.py",
        "-c", str(CONFIG_DST.relative_to(SAM2_CLONE_DIR / "sam2")),
        "--use-cluster", "0", "--num-gpus", "1",
        cwd=SAM2_CLONE_DIR)
else:
    print("Comando a correr manualmente (o poné EJECUTAR_ENTRENAMIENTO=True):\n")
    print(f"cd {SAM2_CLONE_DIR}")
    print(f"python training/train.py -c {CONFIG_DST.relative_to(SAM2_CLONE_DIR / 'sam2')} "
          f"--use-cluster 0 --num-gpus 1")
    print(f"\n# monitoreo:\ntensorboard --logdir {SAM2_CLONE_DIR / 'sam2_logs'}")

## 14 — Validar antes de subir a producción

La loss de training no dice si mejoró en la práctica. Corré el checkpoint
nuevo sobre los frames de `val/` (que nunca vio en entrenamiento) y comparalo
contra el viejo — prestando especial atención a los frames que el modelo
**viejo** ya acertaba (el fallo clásico de un fine-tune chico).

In [ ]:
CHECKPOINT_NUEVO = None   # <-- ruta al .pt que salio de sam2_logs/.../checkpoints/

if CHECKPOINT_NUEVO:
    val_frames = sorted((FOLD_DIR / "val" / "img_folder").glob("*/00000.jpg"))
    print(f"{len(val_frames)} frames de validacion disponibles")
    if val_frames:
        import os
        env = {**os.environ, "SAMTP_CHECKPOINT": str(CHECKPOINT_NUEVO)}
        muestra = val_frames[0]
        subprocess.run(
            [sys.executable, "-m", "rover_traversability.demo", "predict",
             str(muestra), "--out", str(WORK / "check_val0.png")],
            env=env, check=False,
        )
        print(f"Overlay de prueba -> {WORK / 'check_val0.png'}")
else:
    print("CHECKPOINT_NUEVO no seteado todavia -- completalo cuando termine el entrenamiento")

## 15 — Notas finales

- **GPU:** el config de referencia es resolución 1024, batch 8. Una 2080 de
  8 GB no lo aguanta tal cual — bajar `batch_size` o usar cluster/Colab Pro.
- **IoU:** la validación de arriba es a ojo (comparar overlays). Un script
  de IoU real sobre `val/` está marcado como pendiente en
  `1_Doc tecnica y comandos.md` — es ~15 líneas si hace falta armarlo.
- **Shippear el checkpoint** (cuando valide bien):

  ```bash
  hf repo create yourteam/samtp-yourteam --repo-type model --private
  hf upload yourteam/samtp-yourteam nuevo.pt checkpoint_finetuned_v2.pt
  export SAMTP_HF_REPO=yourteam/samtp-yourteam
  ```


In [ ]:
print("Resumen de la corrida\n" + "="*40)
for nombre, ruta in [("metadata", META_DIR), ("selected", SELECTED), ("raw", RAW_DIR),
                     ("clean", CLEAN_DIR), ("index", INDEX), ("curated", CURATED),
                     ("candidatos", CANDIDATOS), ("dataset final", FOLD_DIR)]:
    p = Path(ruta)
    estado = "OK" if p.exists() else "falta"
    print(f"  {nombre:14s} {estado:5s}  {p}")